# RQ2 — Resolution Strategies

> *How are resolution decisions distributed across conflicting chunks
> (V1, V2, CC, CB, NC, NN, Imprecise), and how does this distribution
> compare to the one reported by Ghiotto et al. for human-authored
> merges?* — PLAN.md §3 (RQ2)

Strategy labels emitted by `identify_resolution` are folded to the
seven-bucket scheme of PLAN.md §5.4 (ConcatV1V2/ConcatV2V1 → CC,
Combination → CB, "New code" → NC, "None" → NN, Postponed → Imprecise).

**Imprecise handling.** Ghiotto et al. do not have an `Imprecise`
bucket, so the distribution we report for the paper is *conditional on
chunks we could classify* (i.e., excluding `Imprecise`). For
transparency, this notebook also produces the *unconditional* view
(with `Imprecise`) and reports the fraction of chunks discarded. Two
figure variants are saved for each cut: `..._incl_imprecise` and
`..._excl_imprecise`; the `excl` variant is the one referenced from
the paper.

Figures are saved to `analysis/figures/` as `.pdf` + `.png`.


In [ ]:
# Ensure the project root is on sys.path so ``analysis.common`` imports
# cleanly whether the kernel was launched from the repo root or from
# inside analysis/.
import sys
from pathlib import Path

_here = Path.cwd()
for candidate in [_here, *_here.parents]:
    if (candidate / "analysis" / "common.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from analysis.common import (
    load_tables,
    build_chunk_frame, build_merge_frame, build_pr_frame,
    setup_style, save_fig,
    stratify, stratum_order,
    descriptive_table, strategy_distribution, imprecise_share,
    STRATEGY_ORDER, STRATEGY_PALETTE,
    RESOLVER_ORDER, RESOLVER_PALETTE,
    PR_OUTCOME_ORDER, PR_OUTCOME_PALETTE,
    TOP_N_LANGUAGES,
)

setup_style()
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

# The canonical strategy order excluding Imprecise. Used for paper
# figures and tables, which report the distribution conditional on
# chunks being classifiable.
STRATEGY_ORDER_EXCL = [s for s in STRATEGY_ORDER if s != "Imprecise"]


## 1. Load data & build the chunk frame

In [ ]:
tables = load_tables()          # dedup by (repo, merge_sha) and chunk key
chunks = build_chunk_frame(tables)
print(f"Chunks (classified, deduplicated): {len(chunks):,}")
if not chunks.empty:
    print("\nRaw strategy labels before folding:")
    print(chunks["strategy_raw"].value_counts().to_string())
    imp_share = imprecise_share(chunks)
    print(f"\nImprecise share (global): {imp_share*100:.2f}% of all classified chunks")
    print(f"Classifiable chunks (excluding Imprecise): "
          f"{(chunks['strategy'] != 'Imprecise').sum():,}")


## 2. Global strategy distribution

We report two views:

* **Including Imprecise** (the unconditional chunk distribution).
* **Excluding Imprecise** (the distribution among chunks the
  localization algorithm classified with confidence). This is the
  view used in the paper for comparison with Ghiotto et al.


In [ ]:
# Full view (including Imprecise)
global_dist_incl = strategy_distribution(chunks)
print("Including Imprecise:")
print((global_dist_incl.round(4) * 100).to_string())

# Paper view (excluding Imprecise, percentages re-normalise to 100% over
# the six classifiable buckets)
global_dist_excl = strategy_distribution(chunks, exclude_imprecise=True)
print("\nExcluding Imprecise (paper view):")
print((global_dist_excl.round(4) * 100).to_string())

imp_pct_global = imprecise_share(chunks) * 100
print(f"\nImprecise share excluded from the paper view: {imp_pct_global:.2f}%")


## 3. Figure 1 — Global strategy distribution vs. Ghiotto et al.

Side-by-side bar chart; the Ghiotto percentages are the ones reported
in TSE 2020, Figure 10 (aggregated 2,731 Java projects). Because
Ghiotto does **not** have an `Imprecise` bucket, the paper figure
uses the *excluding-Imprecise* view so the two bars are on the same
support (V1, V2, CC, CB, NC, NN; each set sums to 100%). We save both
variants for completeness.


In [ ]:
GHIOTTO_HUMAN = {
    "V1": 30.14,
    "V2": 26.95,
    "CC": 10.73,
    "CB":  3.58,
    "NC": 21.86,
    "NN":  6.74,
    "Imprecise": 0.00,   # not reported in Ghiotto; kept as 0 for alignment
}

def _plot_vs_ghiotto(order, agent_dist_row, title_suffix, fname, annotate_imprecise=None):
    agent_pct = (agent_dist_row.reindex(order).fillna(0) * 100)
    human_pct = pd.Series({k: GHIOTTO_HUMAN[k] for k in order})
    x = np.arange(len(order))
    w = 0.4
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    ax.bar(x - w/2, agent_pct.values, width=w,
           color=[STRATEGY_PALETTE[s] for s in order],
           edgecolor="black", linewidth=0.5, label="Agents (this study)")
    ax.bar(x + w/2, human_pct.values, width=w,
           color="white", edgecolor="black", hatch="///",
           linewidth=0.5, label="Humans (Ghiotto et al. 2020)")
    ax.set_xticks(x)
    ax.set_xticklabels(order)
    ax.set_ylabel("% of chunks")
    ax.set_title(f"RQ2 — Resolution-strategy distribution: agents vs. humans ({title_suffix})")
    ax.legend(loc="upper right")
    for i, v in enumerate(agent_pct.values):
        ax.text(x[i] - w/2, v + 0.8, f"{v:.1f}", ha="center", fontsize=8)
    for i, v in enumerate(human_pct.values):
        ax.text(x[i] + w/2, v + 0.8, f"{v:.1f}", ha="center", fontsize=8)
    if annotate_imprecise is not None:
        ax.text(0.99, 0.98,
                f"Imprecise chunks excluded: {annotate_imprecise:.1f}%",
                transform=ax.transAxes, ha="right", va="top", fontsize=8,
                color="dimgray")
    save_fig(fig, fname)
    plt.show()

if chunks.empty:
    print("No classified chunks yet -- run the pipeline first.")
else:
    # Paper variant: excluding Imprecise
    _plot_vs_ghiotto(
        STRATEGY_ORDER_EXCL,
        global_dist_excl.iloc[0],
        "Imprecise excluded",
        "rq2_global_vs_ghiotto_excl_imprecise",
        annotate_imprecise=imp_pct_global,
    )
    # Full variant: including Imprecise (for reviewers / appendix)
    _plot_vs_ghiotto(
        STRATEGY_ORDER,
        global_dist_incl.iloc[0],
        "Imprecise included",
        "rq2_global_vs_ghiotto_incl_imprecise",
    )


## 4. Figure 2 — Strategy distribution by agent (stacked bars)

Each column is an agent; segment colours are the canonical strategies.
The paper figure excludes `Imprecise`; the per-agent Imprecise share
is printed below as a small table so it can still be discussed in the
text.


In [ ]:
def _plot_stacked(df, order, palette, title, xlabel, fname, ylim=(0,100)):
    fig, ax = plt.subplots(figsize=(max(5, 0.9 * len(df) + 2), 3.6))
    (df * 100).plot(
        kind="bar", stacked=True,
        color=[palette[s] for s in order],
        edgecolor="white", linewidth=0.3, ax=ax,
    )
    ax.set_ylabel("% of chunks")
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    ax.set_ylim(*ylim)
    ax.legend(title="Strategy", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    save_fig(fig, fname)
    plt.show()

if not chunks.empty and "agent" in chunks.columns:
    agent_order_by_n = chunks["agent"].fillna("Unknown").value_counts().index

    # Paper view (excluding Imprecise)
    by_agent_excl = strategy_distribution(chunks, group_col="agent", exclude_imprecise=True)
    by_agent_excl = by_agent_excl.reindex(agent_order_by_n.intersection(by_agent_excl.index))
    _plot_stacked(by_agent_excl, STRATEGY_ORDER_EXCL, STRATEGY_PALETTE,
                  "RQ2 — Strategy distribution by agent (Imprecise excluded)",
                  "", "rq2_by_agent_excl_imprecise")

    # Full view (including Imprecise)
    by_agent_incl = strategy_distribution(chunks, group_col="agent")
    by_agent_incl = by_agent_incl.reindex(agent_order_by_n.intersection(by_agent_incl.index))
    _plot_stacked(by_agent_incl, STRATEGY_ORDER, STRATEGY_PALETTE,
                  "RQ2 — Strategy distribution by agent (Imprecise included)",
                  "", "rq2_by_agent_incl_imprecise")

    print("Chunks per agent:")
    print(chunks["agent"].value_counts().to_string())
    print("\nImprecise share per agent (%):")
    print((imprecise_share(chunks, group_col="agent") * 100).round(2).sort_values(ascending=False).to_string())


## 5. Figure 3 — Strategy distribution by language (top-N)

In [ ]:
if not chunks.empty and "language_top" in chunks.columns:
    lang_order_by_n = chunks["language_top"].fillna("Unknown").value_counts().index

    by_lang_excl = strategy_distribution(chunks, group_col="language_top", exclude_imprecise=True)
    by_lang_excl = by_lang_excl.reindex(lang_order_by_n.intersection(by_lang_excl.index))
    _plot_stacked(by_lang_excl, STRATEGY_ORDER_EXCL, STRATEGY_PALETTE,
                  "RQ2 — Strategy distribution by language (Imprecise excluded)",
                  "Language", "rq2_by_language_excl_imprecise")

    by_lang_incl = strategy_distribution(chunks, group_col="language_top")
    by_lang_incl = by_lang_incl.reindex(lang_order_by_n.intersection(by_lang_incl.index))
    _plot_stacked(by_lang_incl, STRATEGY_ORDER, STRATEGY_PALETTE,
                  "RQ2 — Strategy distribution by language (Imprecise included)",
                  "Language", "rq2_by_language_incl_imprecise")

    print("\nImprecise share per language (%):")
    print((imprecise_share(chunks, group_col="language_top") * 100).round(2).sort_values(ascending=False).to_string())


## 6. Figure 4 — Strategy distribution by PR task type

In [ ]:
if not chunks.empty and "pr_task_type" in chunks.columns:
    subset = chunks.dropna(subset=["pr_task_type"]).copy()
    if subset.empty:
        print("pr_task_type is only populated under --pop-only.")
    else:
        task_order_by_n = subset["pr_task_type"].value_counts().index

        by_task_excl = strategy_distribution(subset, group_col="pr_task_type", exclude_imprecise=True)
        by_task_excl = by_task_excl.reindex(task_order_by_n.intersection(by_task_excl.index))
        _plot_stacked(by_task_excl, STRATEGY_ORDER_EXCL, STRATEGY_PALETTE,
                      "RQ2 — Strategy distribution by PR task type (Imprecise excluded)",
                      "PR task type", "rq2_by_task_type_excl_imprecise")

        by_task_incl = strategy_distribution(subset, group_col="pr_task_type")
        by_task_incl = by_task_incl.reindex(task_order_by_n.intersection(by_task_incl.index))
        _plot_stacked(by_task_incl, STRATEGY_ORDER, STRATEGY_PALETTE,
                      "RQ2 — Strategy distribution by PR task type (Imprecise included)",
                      "PR task type", "rq2_by_task_type_incl_imprecise")

        print("\nImprecise share per PR task type (%):")
        print((imprecise_share(subset, group_col="pr_task_type") * 100).round(2).sort_values(ascending=False).to_string())


## 7. Per-stratum numerical tables

In [ ]:
if not chunks.empty:
    for axis in ("agent", "language_top", "pr_task_type"):
        if axis not in chunks.columns:
            continue
        # Paper view
        tbl_excl = strategy_distribution(chunks, group_col=axis, exclude_imprecise=True) * 100
        tbl_excl["n_classifiable"] = chunks[chunks["strategy"] != "Imprecise"].groupby(axis).size()
        tbl_excl["imprecise_pct"] = (imprecise_share(chunks, group_col=axis) * 100).round(2)
        tbl_excl["n_total"] = chunks.groupby(axis).size()
        tbl_excl = tbl_excl.sort_values("n_total", ascending=False).round(2)
        print(f"\n=== Strategy % by {axis} (Imprecise excluded) ===")
        print(tbl_excl.to_string())

        # Full view
        tbl_incl = strategy_distribution(chunks, group_col=axis) * 100
        tbl_incl["n"] = chunks.groupby(axis).size()
        tbl_incl = tbl_incl.sort_values("n", ascending=False).round(2)
        print(f"\n=== Strategy % by {axis} (Imprecise included) ===")
        print(tbl_incl.to_string())


---
### Outputs summary

Figures saved to `analysis/figures/` (two variants each):

| File stem                                    | Paper role |
|----------------------------------------------|------------|
| `rq2_global_vs_ghiotto_excl_imprecise`       | **Headline figure (paper)** — agent vs. human side-by-side, excluding `Imprecise` |
| `rq2_global_vs_ghiotto_incl_imprecise`       | Audit variant — includes `Imprecise` |
| `rq2_by_agent_excl_imprecise`                | **Paper** — stratified by agent |
| `rq2_by_agent_incl_imprecise`                | Audit |
| `rq2_by_language_excl_imprecise`             | **Paper** — stratified by language (top-N) |
| `rq2_by_language_incl_imprecise`             | Audit |
| `rq2_by_task_type_excl_imprecise`            | **Paper** — stratified by `pr_task_type` (AIDev-pop only) |
| `rq2_by_task_type_incl_imprecise`            | Audit |

The paper reports the `excl_imprecise` view and the Imprecise share
separately (globally and per stratum), so readers can tell how many
chunks were dropped from the classifiable distribution.
